In [ ]:
from docling.chunking import HybridChunker
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
from transformers import AutoTokenizer

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption

pdf_options = PdfPipelineOptions()
pdf_options.do_ocr = False


EXPORT_TYPE = ExportType.DOC_CHUNKS

EMBED_MODEL_ID = "BAAI/bge-m3" #"BAAI/bge-large-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)

FILE_PATH = "../data/Manuals/mds_axis_parameter_en.pdf"

loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
    chunker=HybridChunker(
        tokenizer=tokenizer,
        max_tokens=700,
        merge_peers=False,
        repeat_table_header=True,
        omit_header_on_overflow=True,
    ),
    converter=DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pdf_options,
                    backend=PyPdfiumDocumentBackend,
                )
            }
        ),
)

c:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs = loader.load()

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 8658.65it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


In [ ]:
def is_useless_chunk(doc) -> bool:
    text = doc.page_content.strip()
    dl_meta = doc.metadata.get("dl_meta", {})

    headings = dl_meta.get("headings", [])
    headings = [h.strip().lower() for h in headings]

    # Supprimer les chunks de table des matières ou index "P"
    if "table of contents" in headings: #overview, P, index, keyword, Appendix, List of figures, Contents, Preface. General and safety instructions, 
        return True

    #if "p" in headings:
    #    return True

    # Supprimer les chunks détectés par Docling comme index
    doc_items = dl_meta.get("doc_items", [])
    for item in doc_items:
        if item.get("label") == "document_index":
            return True

    # Sécurité si le texte commence directement par ces titres
    if text.lower().startswith("table of contents"):
        return True

    if text.startswith("P\n"):
        return True

    return False

In [63]:
import re

def clean_docling_text(text: str) -> str:
    # Remove bad parsed PDF links like: [ } 12], [ } 13], [ } 14]
    text = re.sub(r"\[\s*}\s*\d+\s*\]", "", text)

    # Remove duplicated spaces caused by deletion
    text = re.sub(r"[ \t]{2,}", " ", text)

    return text.strip()

In [64]:
def clean_metadata(doc):
    metadata = doc.metadata
    dl_meta = metadata.get("dl_meta", {})

    filename = dl_meta.get("origin", {}).get("filename")

    page_no = None
    doc_items = dl_meta.get("doc_items", [])

    if doc_items:
        prov = doc_items[0].get("prov", [])
        if prov:
            page_no = prov[0].get("page_no")
    
    heading = dl_meta.get("headings")

    return {
        "filename": filename,
        "page_no": page_no,
        'heading': heading,
    }


m = clean_metadata(docs[78])
print(m)

{'filename': 'mds_axis_compensation_en.pdf', 'page_no': 21, 'heading': ['COMP-00033)']}


In [65]:
docs = [doc for doc in docs if not is_useless_chunk(doc)]

for doc in docs:
    doc.page_content = clean_docling_text(doc.page_content)
    doc.metadata = clean_metadata(doc)

In [71]:
for d in docs:
    print("--------------------------------------------------------------------------------------------")
    print(f"- {d.page_content}")
    print("/n")


--------------------------------------------------------------------------------------------
- Short Description: COMP
© Copyright ISG Industrielle Steuerungstechnik GmbH STEP, Gropiusplatz 10 D-70563 Stuttgart All rights reserved www.isg-stuttgart.de support@isg-stuttgart.de
/n
--------------------------------------------------------------------------------------------
- Legal information
This documentation was produced with utmost care. The products and scope of functions described are under continuous development. We reserve the right to revise and amend the documentation at any time and without prior notice.
/n
--------------------------------------------------------------------------------------------
- Legal information
No claims may be made for products which have already been delivered if such claims are based on the specifications, figures and descriptions contained in this documentation.
/n
--------------------------------------------------------------------------------------